In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.interpolate import griddata
from src.real_data.quotes import load, clean, forwards, invert

raw = load(REPO / "datasets/raw/spxw/2026-07-08.dbn.zst", "2026-07-08")
snaps = np.sort(raw["ts_recv"].unique())
EOD = snaps[-1]

def surface(ts):
    q = clean(raw[raw.ts_recv == ts])
    return invert(q, forwards(q))

In [ ]:
d = raw[raw.ts_recv == EOD]
print(f"EOD {pd.Timestamp(EOD):%Y-%m-%d %H:%M}   {len(d):,} quotes")
print(f"crossed (ask < bid): {(d.ask < d.bid).sum()}")
print(f"zero/nan bid:        {((d.bid <= 0) | d.bid.isna()).sum()}")
print(f"below $0.10:         {(d.mid < 0.10).sum()}")
print(f"clean two-sided:     {((d.bid > 0) & (d.ask >= d.bid) & (d.mid >= 0.10)).mean():.1%}")

In [ ]:
q = surface(EOD)
dom = q[(q.z >= -1.5) & (q.z <= 0.5)]

def surf3d(ax, x, y, iv, xg, yg, xlab, title):
    if xg is None:
        ax.plot_trisurf(x, y, iv, cmap="turbo", edgecolor="none", alpha=0.9)
    else:
        XG, YG = np.meshgrid(xg, yg)
        ax.plot_surface(XG, YG, griddata((x, y), iv, (XG, YG)), cmap="viridis", edgecolor="none")
    ax.set(xlabel=xlab, ylabel="t (yr)", zlabel="IV", title=title); ax.view_init(25, -60)

kg = np.linspace(q.k.min(), q.k.max(), 60)
zg = np.linspace(-1.5, 0.5, 60)
tg = np.linspace(q.tau.min(), min(q.tau.max(), 1.0), 60)

fig = plt.figure(figsize=(13, 10))
surf3d(fig.add_subplot(221, projection="3d"), q.k, q.tau, q.mid_iv, None, None, "k", "physical raw")
surf3d(fig.add_subplot(222, projection="3d"), q.k, q.tau, q.mid_iv, kg, tg, "k", "physical interp")
surf3d(fig.add_subplot(223, projection="3d"), dom.z, dom.tau, dom.mid_iv, None, None, "z", "domain raw")
surf3d(fig.add_subplot(224, projection="3d"), dom.z, dom.tau, dom.mid_iv, zg, tg, "z", "domain interp")
fig.tight_layout()

In [ ]:
picks = {"morning": snaps[20], "midday": snaps[len(snaps) // 2], "close": EOD}
ZG, TG = np.meshgrid(np.linspace(-1.5, 0.5, 60), np.linspace(0.02, 1.0, 60))

grids = {}
fig = plt.figure(figsize=(15, 4.5))
for i, (name, ts) in enumerate(picks.items()):
    d = surface(ts); d = d[(d.z > -1.6) & (d.z < 0.6)]
    grids[name] = griddata((d.z, d.tau), d.mid_iv, (ZG, TG))
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    ax.plot_surface(ZG, TG, grids[name], cmap="viridis", edgecolor="none", vmin=0.1, vmax=0.6)
    ax.set(xlabel="z", ylabel="t", zlabel="IV", title=f"{name} {pd.Timestamp(ts):%H:%M}", zlim=(0.1, 0.6))
    ax.view_init(25, -60)
fig.tight_layout()

m = np.isfinite(grids["morning"]) & np.isfinite(grids["midday"]) & np.isfinite(grids["close"])
base = grids["morning"][m]
for k in ["midday", "close"]:
    v = grids[k][m]
    corr = np.corrcoef(base - base.mean(), v - v.mean())[0, 1]
    print(f"{k:>7} vs morning:  shape-corr {corr:.4f}   level {100 * (v.mean() - base.mean()):+.2f}pp")